# Lab 2: 검색(Retrieval) + 생성(Generation) 파이프라인 구축

RAG(Retrieval-Augmented Generation)의 두 번째 단계입니다.
ChromaDB에서 검색한 문서를 LLM에 컨텍스트로 전달하여 정확한 답변을 생성합니다.

**학습 목표:**
- 검색(Retrieval) + 생성(Generation) 파이프라인 통합
- Mock LLM으로 RAG 흐름 이해
- 실제 LLM 연동을 위한 코드 구조 파악

In [ ]:
# 이전 Lab에서 ChromaDB와 임베딩 모델 재사용
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.Client()

# 컬렉션이 이미 존재하면 가져오고, 없으면 생성
try:
    collection = client.get_collection("manufacturing_docs")
    print(f"기존 컬렉션 로드: {collection.count()}개 문서")
except Exception:
    collection = client.create_collection("manufacturing_docs")
    documents = [
        {"id": "doc1", "text": "CNC 머신 주축 베어링 이상 진동 발생 시 즉시 가동 중단 후 점검 필요. 진동값 3mm/s 초과 시 위험 신호.", "metadata": {"source": "설비매뉴얼", "topic": "베어링"}},
        {"id": "doc2", "text": "용접부 품질 기준: 인장강도 400MPa 이상, 경도 HRC 20~30. 초음파 검사 주기: 월 1회.", "metadata": {"source": "품질기준서", "topic": "용접"}},
        {"id": "doc3", "text": "스마트공장 ROI 사례: 생산성 33.6% 향상, 품질 불량률 44.4% 감소. 초기 투자 2억 → 3년 회수.", "metadata": {"source": "KAMP사례집", "topic": "ROI"}},
        {"id": "doc4", "text": "AutoEncoder 기반 이상탐지: 정상 데이터만 학습, 재구성 오차가 임계값 초과 시 이상 판정. 민감도 조정 가능.", "metadata": {"source": "AI가이드", "topic": "이상탐지"}},
    ]
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    embeddings = model.encode([d["text"] for d in documents]).tolist()
    collection.add(
        ids=[d["id"] for d in documents],
        embeddings=embeddings,
        documents=[d["text"] for d in documents],
        metadatas=[d["metadata"] for d in documents]
    )
    print(f"신규 컬렉션 생성: {collection.count()}개 문서")

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# LangChain RAG 파이프라인 (mock LLM 사용)
from langchain.schema import Document

class MockManufacturingLLM:
    """제조 도메인 특화 응답 시뮬레이터 (실제 LLM 대체)"""
    def generate(self, query, context_docs):
        context = "\n".join([d["text"] for d in context_docs])
        return f"""[제조 AI 응답]
질문: {query}

관련 지식 기반 검색 결과:
{context[:200]}...

현장 조치 권고: 위 기준에 따라 즉각 점검 스케줄을 수립하세요."""

llm = MockManufacturingLLM()

def rag_pipeline(query, collection, model, top_k=2):
    """RAG 파이프라인 실행"""
    # 1. 검색 (Retrieval)
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=top_k)
    context_docs = [
        {"text": doc, "meta": meta}
        for doc, meta in zip(results['documents'][0], results['metadatas'][0])
    ]
    
    # 2. 생성 (Generation)
    response = llm.generate(query, context_docs)
    
    return {"query": query, "context": context_docs, "response": response}

# 첫 번째 테스트
result = rag_pipeline("베어링 진동 기준 알려줘", collection, model)
print(result["response"])
print("\n[검색된 컨텍스트 문서]")
for i, doc in enumerate(result["context"]):
    print(f"  [{i+1}] {doc['meta']['source']} - {doc['text'][:60]}...")

In [ ]:
# 여러 질문으로 RAG 테스트 루프
test_queries = [
    "스마트공장 도입하면 얼마나 효과가 있어?",
    "AutoEncoder로 이상탐지 어떻게 해?",
    "용접 품질 검사는 얼마나 자주 해야 해?",
]

print("=" * 60)
print("RAG 파이프라인 테스트 결과")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
    result = rag_pipeline(query, collection, model)
    print(f"\n[Q{i}] {query}")
    print(f"검색 문서: {[doc['meta']['topic'] for doc in result['context']]}")
    print(f"응답 미리보기: {result['response'][:120]}...")
    print("-" * 40)

## 핵심 요약

**RAG = 검색(문서DB) + 생성(LLM) = 환각 없는 정확한 답변**

```
사용자 질문
    ↓
[검색 단계] 질문 → 임베딩 → ChromaDB 유사도 검색 → 관련 문서 k개
    ↓
[생성 단계] 관련 문서 + 질문 → LLM → 근거 있는 답변
    ↓
최종 응답 (출처 명시 가능)
```

**왜 RAG인가?**
| 기존 LLM | RAG |
|---|---|
| 학습 데이터 기반 답변 | 실시간 문서 기반 답변 |
| 환각(Hallucination) 위험 | 검색 문서로 근거 제시 |
| 도메인 업데이트 어려움 | 문서 추가로 즉시 반영 |

**다음 단계:** Streamlit UI로 챗봇 인터페이스 구축 (`src/streamlit_app.py`)